# 02 — ML Risk Model (Heart Disease)

Gradient-boosting classifier for binary heart-disease prediction.

**Lineage.**
- Scikit-learn `Pipeline` + scaling discipline → Project 3 (UCI Online Retail II K-Means RFM).
- Per-slice disaggregated evaluation by `sex` and age band → Project 4 (Fashion-MNIST CNN with dropout), where same headline accuracy hid large per-class differences.

**Educational artifact only. Not for clinical use.**

In [ ]:
# Notebook bootstrap: import path setup and bring in the project modules used below.

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, precision_recall_curve, confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance

from src.data_loader import load_heart_disease
from src.preprocessing import split_and_preprocess
from src.ml_model import train_and_evaluate, predict_proba, save, slice_metrics

## 1. Train + cross-validate

In [ ]:
# End-to-end training: load data -> stratified train/test split with the same
# sklearn ColumnTransformer used in production -> train the gradient-boosting
# classifier (HistGradientBoosting) -> evaluate on the held-out test set ->
# persist the fitted model and preprocessor to models/ for downstream notebooks.

df = load_heart_disease()
split = split_and_preprocess(df)
model, metrics = train_and_evaluate(split)
print(metrics)
print('Saved model artifact:', save(model).name)


## 2. ROC and Precision-Recall curves

In [ ]:
# Compute test-set predicted probabilities, then derive ROC and PR curves.
# AUC and average-precision summarise the trade-off space across all thresholds.

probs = predict_proba(model, split.X_test)
fpr, tpr, _ = roc_curve(split.y_test, probs)
prec, rec, _ = precision_recall_curve(split.y_test, probs)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title(f'ROC (AUC = {metrics.test_roc_auc:.3f})')
axes[1].plot(rec, prec)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR (AUC = {metrics.test_pr_auc:.3f})')
plt.tight_layout(); plt.show()

<!-- chart-narration -->
**Reading the ROC and PR curves.**
- **ROC** plots true-positive rate vs. false-positive rate as the decision threshold sweeps from 1.0 down to 0.0. The diagonal is a random classifier; the top-left corner is perfect. AUC near 0.96 means: for a random (positive, negative) pair, the model ranks the positive higher 96% of the time. Good.
- **Precision-Recall** is more informative than ROC under class imbalance. Average precision ~0.95 with the curve hugging the top-right means we retain high precision even at high recall. This is the operating regime a triage system needs.

What would be bad: an ROC that hugs the diagonal (random), or a PR curve that collapses at high recall (cannot find positives without flooding the queue with false positives).

## 3. Confusion matrix at threshold 0.5

In [ ]:
# Confusion matrix at the conventional 0.5 decision threshold. The classification
# report exposes per-class precision/recall/F1 - the disaggregated view we need
# for a clinical triage tool where false-negative cost > false-positive cost.

y_pred = (probs >= 0.5).astype(int)
cm = confusion_matrix(split.y_test, y_pred)
print(cm)
print(classification_report(split.y_test, y_pred, digits=3))

<!-- chart-narration -->
**Reading the confusion matrix and classification report.** At threshold 0.5:
- True negatives + true positives along the diagonal = correct predictions.
- The off-diagonals show error structure. In a clinical-triage prototype, the costlier error is a **false negative** (sick patient told they are not at risk). The per-class recall row in the classification report tells you how often each true class is caught.
- Macro-F1 averages performance across both classes equally; weighted-F1 weights by class size. With near-balanced classes the two are close, so any large gap would point to a single-class collapse.

## 4. Permutation feature importance
Permutation importance is computed on the held-out test set so we measure what the model uses to generalise, not what it used to fit.

In [ ]:
# Permutation importance on the held-out test set, scored by ROC-AUC.
# This is the post-hoc, model-agnostic attribution: how much does test
# performance degrade when each feature column is randomly shuffled?

result = permutation_importance(
    model, split.X_test, split.y_test, n_repeats=20, random_state=42, scoring='roc_auc'
)
imp = pd.DataFrame({
    'feature': split.X_test.columns,
    'importance': result.importances_mean,
    'std': result.importances_std,
}).sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(imp['feature'], imp['importance'], xerr=imp['std'], color='#4C9AFF')
ax.set_xlabel('Drop in ROC-AUC when shuffled')
ax.set_title('Permutation importance (test set)')
plt.tight_layout(); plt.show()

<!-- chart-narration -->
**Reading permutation importance.** Each bar is the drop in test ROC-AUC when that feature's column is randomly shuffled (`n_repeats=20`, scoring=`roc_auc`). The order of bars is the model's actual decision-making attribution on held-out data, *not* training-time feature importance (which is biased by tree splits).

Expected top features for this cohort: `cp` (chest-pain type), `thal` (perfusion defect), `ca` (vessel count), `oldpeak` (ST depression). These match the EDA findings in notebook 01, which is the consistency check we want before trusting the model.

## 5. Per-slice metrics — bias audit
Lineage: P4 surfaced that aggregate accuracy can hide large per-class differences (Coat +9.4, Shirt -6.1 between identical-headline models). The same disaggregation discipline applied here checks whether the headline ROC-AUC hides cohort-level disparities.

In [ ]:
# Per-slice metrics: ROC-AUC computed separately for each subgroup.
# Lineage from Project 4: "aggregate accuracy can hide large per-class gaps".
# Here we audit sex and age-band slices for the same kind of hidden bias.

print('--- By sex (0 = female, 1 = male) ---')
print(slice_metrics(model, split.X_test, split.y_test, 'sex'))

In [ ]:
# Age-band slice with clinically sensible cuts (<45, 45-55, 55-65, 65+).

# Age-band slice
X_test_aug = split.X_test.copy()
X_test_aug['age_band'] = pd.cut(
    X_test_aug['age'], bins=[0, 45, 55, 65, 120],
    labels=['<45', '45-54', '55-64', '65+']
)
print('--- By age band ---')
print(slice_metrics(model, X_test_aug, split.y_test, 'age_band'))

## 6. Notes for the synthesis paper
- Headline ROC-AUC is strong on this small public cohort, but slice sizes are tiny — slice AUCs are point estimates with wide implicit confidence bands.
- The disaggregation discipline is the point, not the headline number; it carries forward into Phase J (Evaluation).
- The trained model is persisted under `models/ml_model.joblib` (gitignored) and consumed by the agent orchestrator in Phase H.